# 02 - 采集日线数据

**功能**: 从 Tushare 采集个股日线数据（5 年历史）

In [ ]:
import os
import pandas as pd
import tushare as ts
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv
from tqdm import tqdm

# 初始化
PROJECT_ROOT = Path('../../').resolve()
load_dotenv(PROJECT_ROOT / '.env')
TUSHARE_TOKEN = os.getenv('TUSHARE_TOKEN')

ts.set_token(TUSHARE_TOKEN)
pro = ts.pro_api()

db_path = PROJECT_ROOT / 'data' / 'invest.db'
conn = sqlite3.connect(db_path)
print("✅ 初始化完成")

In [ ]:
# 选择测试股票（可以修改）
test_stocks = [
    '000001.SZ',  # 平安银行
    '000002.SZ',  # 万科 A
    '600036.SH',  # 招商银行
    '600519.SH',  # 贵州茅台
    '300750.SZ',  # 宁德时代
]

# 设置时间范围（5 年）
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=365*5)).strftime('%Y%m%d')

print(f"采集时间范围：{start_date} 至 {end_date}")
print(f"测试股票：{', '.join(test_stocks)}")

In [ ]:
# 采集日线数据
all_daily = []

for code in tqdm(test_stocks, desc="采集日线"):
    try:
        df = pro.daily(ts_code=code, start_date=start_date, end_date=end_date)
        if len(df) > 0:
            all_daily.append(df)
    except Exception as e:
        print(f"❌ {code} 采集失败：{e}")

if all_daily:
    daily_df = pd.concat(all_daily, ignore_index=True)
    print(f"\n✅ 采集完成，共 {len(daily_df)} 条记录")
    
    # 保存到数据库
    daily_df.to_sql('stock_daily', conn, if_exists='replace', index=False)
    print("✅ 数据已保存到数据库")
else:
    print("❌ 没有采集到数据")

In [ ]:
# 可视化：股价走势
import plotly.graph_objects as go

daily_df['trade_date'] = pd.to_datetime(daily_df['trade_date'], format='%Y%m%d')

fig = go.Figure()

for code in test_stocks:
    stock_data = daily_df[daily_df['ts_code'] == code].sort_values('trade_date')
    if len(stock_data) > 0:
        fig.add_trace(go.Scatter(
            x=stock_data['trade_date'],
            y=stock_data['close'],
            mode='lines',
            name=code
        ))

fig.update_layout(
    title='测试股票股价走势（5 年）',
    xaxis_title='日期',
    yaxis_title='收盘价（元）',
    height=600,
    hovermode='x unified'
)

fig.show()